# RoadGuardian AI — detección de animales y vehículos

Notebook para Google Colab. A partir de una fotografía, dibuja cajas, etiquetas y porcentajes de confianza para vehículos y animales detectables por el modelo.

**Cómo se relaciona con tus repositorios:** reutiliza el flujo de cargar una imagen, detectar objetos, mostrar resultados y contar detecciones del proyecto de carreteras. El clasificador Keras de felinos se diseñó para clasificar la imagen completa entre sus clases de felinos; no localiza cada animal con cajas. Por eso este prototipo usa un detector general para localizar vehículos y animales. Después se puede conectar el clasificador de felinos cuando haya recortes de animales que queramos clasificar.

> Es un prototipo educativo. El nivel de riesgo es una regla visual aproximada y no debe usarse como sistema real de seguridad vial.

In [ ]:
!pip -q install ultralytics opencv-python-headless

## 1. Subir hasta 4 fotografías

Ejecuta la celda siguiente y selecciona hasta cuatro imágenes de carreteras al mismo tiempo.

In [ ]:
from google.colab import files
uploaded = files.upload()
image_paths = list(uploaded.keys())[:4]

if len(image_paths) == 0:
    raise ValueError('No se subieron imágenes.')

if len(uploaded) > 4:
    print('Se seleccionaron más de 4 imágenes; se analizarán las primeras 4.')

print('Imágenes que se analizarán:', len(image_paths))
for image_path in image_paths:
    print('-', image_path)

## 2. Detectar vehículos y animales en las imágenes

El modelo analiza cada fotografía y dibuja cajas con el nombre y la confianza. Verde = animales; azul/naranja = vehículos.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display
from collections import Counter
import cv2
import os

# Modelo pequeño; Colab lo descarga automáticamente la primera vez.
model = YOLO('yolov8n.pt')

vehicle_classes = {'car', 'motorcycle', 'bus', 'truck', 'bicycle'}
animal_classes = {'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe'}

all_summaries = []
for image_path in image_paths:
    print('\n' + '=' * 60)
    print('Analizando:', image_path)
    results = model.predict(source=image_path, conf=0.25, verbose=False)
    result = results[0]
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        print('No se pudo abrir esta imagen; se omite.')
        continue

    counts = Counter()
    animals = []
    vehicles = []
    for box in result.boxes:
        class_id = int(box.cls[0].item())
        label = model.names[class_id]
        confidence = float(box.conf[0].item())
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        if label not in vehicle_classes and label not in animal_classes:
            continue

        counts[label] += 1
        if label in animal_classes:
            animals.append((x1, y1, x2, y2, label, confidence))
            color = (0, 200, 0)  # verde: animales
        else:
            vehicles.append((x1, y1, x2, y2, label, confidence))
            color = (255, 120, 0)  # azul/naranja: vehículos

        cv2.rectangle(image_bgr, (x1, y1), (x2, y2), color, 2)
        text = f'{label}: {confidence * 100:.1f}%'
        cv2.putText(image_bgr, text, (x1, max(22, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    # Mostrar el resultado en Colab; no se descarga ningún archivo.
    display(Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)))
    print('Resumen de detecciones:')
    if counts:
        for name, count in sorted(counts.items()):
            print(f'  {name}: {count}')
    else:
        print('  No se detectaron animales ni vehículos de las clases seleccionadas.')
    print('Animales:', len(animals), '| Vehículos:', len(vehicles))
    if animals and vehicles:
        print('AVISO: hay animales y vehículos en la misma imagen; revisa visualmente si existe riesgo.')
    all_summaries.append((image_path, counts, len(animals), len(vehicles)))

print('\nAnálisis terminado. Las imágenes anotadas se muestran arriba y no se descargan automáticamente.')

In [ ]:
print('--- RESUMEN DE ROADGUARDIAN AI ---')
if counts:
    for name, count in sorted(counts.items()):
        print(f'{name}: {count}')
else:
    print('No se detectaron animales ni vehículos de las clases seleccionadas.')

print('\nAnimales detectados:', len(animals))
print('Vehículos detectados:', len(vehicles))

if animals and vehicles:
    print('\nAVISO: hay al menos un animal y un vehículo en la misma imagen.')
    print('Revisa visualmente si el animal está en la zona de circulación.')
elif animals:
    print('\nSe detectaron animales, pero no vehículos de las clases seleccionadas.')
elif vehicles:
    print('\nSe detectaron vehículos, pero no animales de las clases seleccionadas.')
else:
    print('\nNo hay detecciones seleccionadas que resumir.')
